In [4]:
import pandas as pd

def clean_silver(df):
    # Drop columns: 'Unnamed: 0', 'Unnamed: 1' and 66 other columns
    df = df.drop(columns=['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 16', 'Unnamed: 15', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 29', 'Unnamed: 28', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37', 'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44', 'Unnamed: 45', 'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48', 'Unnamed: 50', 'Unnamed: 51', 'Unnamed: 52', 'Unnamed: 53', 'Unnamed: 54', 'Unnamed: 55', 'Unnamed: 57', 'Unnamed: 58', 'Unnamed: 59', 'Unnamed: 60', 'Unnamed: 61', 'Unnamed: 62', 'Unnamed: 63', 'Unnamed: 64', 'Unnamed: 65', 'Unnamed: 66', 'Unnamed: 68', 'Unnamed: 67', 'Unnamed: 69', 'Unnamed: 71', 'Unnamed: 70'])
    df = df.drop(df.index[:10])
    # Rename column 'Unnamed: 6' to 'Date'
    df = df.rename(columns={'Unnamed: 6': 'Date'})
    df = df.rename(columns={'Unnamed: 12': 'Order #'})
    df = df.rename(columns={'Unnamed: 49': 'Silver Debit'})
    df = df.rename(columns={'Unnamed: 56': 'Credit'})
    # Convert 'Credit' column to numeric and negate its absolute values
    df['Credit'] = pd.to_numeric(df['Credit'].astype(str).str.replace(',', '', regex=False), errors='coerce')
    df['Credit'] = -df['Credit'].abs()
    df['Silver Debit'] = pd.to_numeric(df['Silver Debit'].astype(str).str.replace(',', '', regex=False), errors='coerce')
    df['Credit'] = df['Credit'].replace(-0, 0)
    # Ensure 'Date' column is in datetime format
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    # Format 'Date' column to desired string format
    df['Date'] = df['Date'].dt.strftime('%m/%d/%Y')
    return df

df = pd.read_excel(r'c:\Users\MichaelGinsberg\Documents\TPM_Development\accounting_reconciling_match\Data\silver_ledger.xlsx')

df_silver = clean_silver(df.copy())
df_silver.head()

,Date,Order #,Silver Debit,Credit
10,01/02/2026,INV MISU,296.40,0.00
11,01/02/2026,INV MISU,1481688.20,0.00
12,01/02/2026,INV MISU,156178.26,0.00
13,01/02/2026,709074 CA: INV MISU,296.40,0.00
14,01/02/2026,709074 CA: INV MRCP,0.00,-296.84


In [5]:
import pandas as pd

def clean_inv(df):
    # Drop columns: 'SITE', 'FROM SITE' and 9 other columns
    df = df.drop(columns=['SITE', 'FROM SITE', 'ACCOUNT', 'UC1', 'UC2', 'UC3', 'UC4', 'JOURNAL', 'Item', 'C/V NUM', 'Name'])
    # Rename column 'Document Number' to 'Order #'
    df = df.rename(columns={'Document Number': 'Order #'})
    # Rename column 'DATE' to 'Date'
    df = df.rename(columns={'DATE': 'Date'})
    # Fix column assignment to DataFrame
    df['Date'] = df['Date'].dt.strftime('%m/%d/%Y')
    # Drop column: 'REFERENCE'
    df = df.drop(columns=['REFERENCE'])
    # Rename column 'AMOUNT' to 'Credit'
    df = df.rename(columns={'AMOUNT': 'Credit'})
    df['Credit'] = pd.to_numeric(df['Credit'], errors='coerce')
    return df

# Loaded variable 'df' from URI: c:\Users\MichaelGinsberg\Documents\TPM_Development\accounting_reconciling_match\Data\Finance_ueGLDetailwItemDataViewExport4_16_2026.xlsx
df = pd.read_excel(r'c:\Users\MichaelGinsberg\Documents\TPM_Development\accounting_reconciling_match\Data\silver_credit.xlsx')

df_inv = clean_inv(df.copy())
df_inv.head()

,Date,Credit,Order #
0,01/02/2026,-1484200.0,XPO042751
1,01/02/2026,-148420.0,XPO042751
2,01/02/2026,-10100.0,XPO042751
3,01/02/2026,-9100.0,XPO042751
4,01/02/2026,296.4,XPO42751


In [ ]:
from collections import defaultdict

# Build lookup: (Date, Credit) -> list of Order #s from df_inv
# Each entry is consumed once so the same Order # is never assigned twice
inv_lookup = defaultdict(list)
for _, row in df_inv.dropna(subset=['Order #']).iterrows():
    inv_lookup[(row['Date'], row['Credit'])].append(row['Order #'])

# Replace Order # in df_silver if either:
#   - Date + Credit match df_inv's Date + Credit, OR
#   - Date + Silver Debit match df_inv's Date + Credit
# Each match is consumed (popped) so it cannot be reused for a second row.
# Falls back to original Order # if no matches remain.
new_order_nums = []
for date, credit, silver_debit, orig in zip(
    df_silver['Date'], df_silver['Credit'], df_silver['Silver Debit'], df_silver['Order #']
):
    if inv_lookup[(date, credit)]:
        new_order_nums.append(inv_lookup[(date, credit)].pop(0))
    elif inv_lookup[(date, silver_debit)]:
        new_order_nums.append(inv_lookup[(date, silver_debit)].pop(0))
    else:
        new_order_nums.append(orig)
df_silver['Order #'] = new_order_nums

df_silver.head()

In [8]:
df_summary = (
    df_silver
    .groupby('Order #', as_index=False)[['Silver Debit', 'Credit']]
    .sum()
)
df_summary['Remainder'] = df_summary['Silver Debit'] + df_summary['Credit']
df_summary

,Order #,Silver Debit,Credit,Remainder
0,709074 CA: INV MRCP,0.00,-296.84,-296.84
1,XPO042260,384.60,0.00,384.60
2,XPO042270,30176.75,0.00,30176.75
3,XPO042332,64.63,0.00,64.63
4,XPO042368,55562.50,0.00,55562.50
...,...,...,...,...
1825,XPO044764,0.00,-2149250.00,-2149250.00
1826,XPO044765,8400.00,-50424.00,-42024.00
1827,XPO044767,0.00,-8413.00,-8413.00
1828,XPO42751,592.80,0.00,592.80


In [9]:
df_grouped = df_silver.groupby('Order #', as_index=False)[['Silver Debit', 'Credit']].sum()
df_grouped['Silver Debit'] = df_grouped['Silver Debit'].round(2)
df_grouped['Credit'] = df_grouped['Credit'].round(2)
df_grouped['Remainder'] = df_grouped['Silver Debit'] + df_grouped['Credit']
df_grouped

,Order #,Silver Debit,Credit,Remainder
0,709074 CA: INV MRCP,0.00,-296.84,-296.84
1,XPO042260,384.60,0.00,384.60
2,XPO042270,30176.75,0.00,30176.75
3,XPO042332,64.63,0.00,64.63
4,XPO042368,55562.50,0.00,55562.50
...,...,...,...,...
1825,XPO044764,0.00,-2149250.00,-2149250.00
1826,XPO044765,8400.00,-50424.00,-42024.00
1827,XPO044767,0.00,-8413.00,-8413.00
1828,XPO42751,592.80,0.00,592.80


In [10]:
df_grouped.to_excel('silver_cleaned.xlsx', index=False)